# Step by step particle dynamics simulations
The aim of this notebook is to build gradually certain particle dynamics regarding its orientation and position.

In [1]:
import sys
from pathlib import Path

import numpy as np 
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Path to src_Jeffery.py
src_path = Path("../Pysimulations").resolve()

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

import src_Jeffery as jf

In [2]:
#Parameters 
import tomli
from pathlib import Path

parameter_file = Path("parameters.toml")

with open(parameter_file, "rb") as f:
    params = tomli.load(f)

print("parameters.toml loaded correctly")

parameters.toml loaded correctly


In [3]:

# PARTICLE PARAMETERS

Length = params["particle"]["length"]
diameter = params["particle"]["diameter"]

swimming_speed = params["particle"]["swimming_speed"]

eta = params["particle"]["eta"]

n_particle = params["particle"]["n_particle"]
n_medium = params["particle"]["n_medium"]


# INITIAL CONDITIONS

R0 = np.array([
    params["initial"]["x0"],
    params["initial"]["y0"],
    params["initial"]["z0"]
], dtype=float)

theta0_deg = params["initial"]["theta0_deg"]
phi0_deg = params["initial"]["phi0_deg"]


# FLOW PARAMETERS

gamma = params["flow"]["gamma"]
plane = params["flow"]["plane"]


# OPTICAL PARAMETERS

wavelength = params["optics"]["wavelength"]
w0 = params["optics"]["w0"]
laser_power = params["optics"]["laser_power"]

beam_axis = np.array(
    params["optics"]["beam_axis"],
    dtype=float
)

e_pol = np.array(
    params["optics"]["polarization"],
    dtype=float
)


# SIMULATION PARAMETERS

t0 = params["simulation"]["t0"]
tf = params["simulation"]["tf"]
n_points = params["simulation"]["n_points"]


print("PARTICLE")
print("Length =", Length, "m")
print("diameter =", diameter, "m")
print("swimming_speed =", swimming_speed, "m/s")
print("eta =", eta, "Pa s")
print("n_particle =", n_particle)
print("n_medium =", n_medium)

print()

print("INITIAL CONDITIONS")
print("R0 =", R0, "m")
print("theta0 =", theta0_deg, "deg")
print("phi0 =", phi0_deg, "deg")

print()

print("FLOW")
print("gamma =", gamma, "1/s")
print("plane =", plane)

print()

print("OPTICS")
print("wavelength =", wavelength, "m")
print("w0 =", w0, "m")
print("laser_power =", laser_power, "W")
print("beam_axis =", beam_axis)
print("polarization =", e_pol)



print()

print("SIMULATION")
print("t0 =", t0, "s")
print("tf =", tf, "s")
print("n_points =", n_points)


PARTICLE
Length = 1e-05 m
diameter = 7e-06 m
swimming_speed = 1e-05 m/s
eta = 0.000953 Pa s
n_particle = 1.38
n_medium = 1.33

INITIAL CONDITIONS
R0 = [0. 0. 0.] m
theta0 = 25.0 deg
phi0 = 25.0 deg

FLOW
gamma = 1e-06 1/s
plane = xy

OPTICS
wavelength = 9.76e-07 m
w0 = 5e-05 m
laser_power = 0.14 W
beam_axis = [0. 0. 1.]
polarization = [0. 1. 0.]

SIMULATION
t0 = 0.0 s
tf = 300.0 s
n_points = 4000


## Properties of the particle
The particle is approximated as a prolate spheroid with total length L and diameter D.

The aspect ratio is 

$$c = \frac{L}{D}$$

and the corresponding Bretherton parameter is 

$$\beta = \frac{c^2-1}{c^2+1}$$

The Bretherton parameter becomes 0 for a sphere ($\beta=0$), approaches 1 at the slender limit ($\beta\rightarrow 1$) and approaches -1 at the disk limit ($\beta\rightarrow-1 $).

The volume is calculated through 

$$V = \frac{4}{3}\pi ab^2$$

Using $a= \text{Length}/2$ and $b= \text{diameter}/2$.

The particle's polarizability can be obtained from Clausius-Mossoti:

$$\alpha=3V\varepsilon_0\frac{\varepsilon_r-1}{\varepsilon_r+2} $$

Using

$$\varepsilon_r= \left(\frac{n_p}{n_m} \right)^2$$

Ishimoto, K. (2023). Jeffery’s orbits and microswimmers in flows: a theoretical review. Journal of the Physical Society of Japan, 92(6), 062001.

Levy, O., & Bergman, D. J. (1992). Clausius-Mossotti approximation for a family of nonlinear composites. Physical Review B, 46(11), 7189.

In [4]:
#Particle Geometry
aspect_ratio = Length/diameter

beta = jf.lambda_ar(aspect_ratio)
print("Bretherton parameter: ",beta)
print("With aspecto ratio of: ",aspect_ratio)

a = Length/2
b = diameter/2

Volume = (4/3)*np.pi*a*(b**2)
print("the volume is:", Volume, "m^3")

alpha_scalar = jf.alpha(Volume, n_particle, n_medium)
print("The particle's polarizability is:", alpha_scalar)

Bretherton parameter:  0.3422818791946309
With aspecto ratio of:  1.4285714285714286
the volume is: 2.565634000431664e-16 m^3
The particle's polarizability is: 1.6967954235295687e-28


## Initial orientation
The orientation of the particle is represented by the unit director 

$$\mathbf{p}=(p_x, p_y, p_z),\qquad ||\mathbf{p}|| = 1 $$

Using spherical angles,

$$p_x = \sin\theta\cos\phi $$
$$p_y = \sin\theta\sin\phi $$
$$p_z = \cos\theta $$

In [5]:
#angle to radians 
theta0 = np.deg2rad(theta0_deg)
phi0 = np.deg2rad(phi0_deg)

p0 = jf.sph_to_vec(theta0, phi0)
print("the initial orientation is p: ", p0)
print("||p0|| =", np.linalg.norm(p0))

the initial orientation is p:  [0.38302222 0.1786062  0.90630779]
||p0|| = 1.0


## Simulation outputs
The dynamics will be presented using the time evolution of the director:

$$ \mathbf{p}(t)=(p_x(t), p_y(t), p_z(t)) $$

Time evolution of the center of mass through time:

$$ \mathbf{R}(t)=(x(t), y(t), z(t)) $$

Angular representation through spherical angles

$$\theta(t),\qquad \phi(t) $$

and the orientator unit vector is presented in a unit sphere, a sphere of radius 1 is parametrized as

$$x=\sin\theta\cos\phi, $$

$$y=\sin\theta\sin\phi,$$

$$z=\cos\theta $$

with

$$0\le \theta \le \pi,\qquad0\le \phi < 2\pi.$$

- The sphere represents the trajectory of $\mathbf{p}$
- Rhe red line represents the rhjectory $\mathbf{p}(t)$
- The green arrow is the initial orientation $p_0$
- The blue arrow is the final position $p_f$ at t
- The orange arrow represents the beam axis
- The purple arrow indicates the polarization direction

For a simple shear flow in the \(xy\)-plane,

$$ \mathbf u=(\dot\gamma y,0,0) $$

The black arrows represent samples of this velocity field.

In [6]:
#Background flow

G = jf.grad_u_simple_shear(gamma,plane)

E, W = jf.decompose_grad_u(G)

print(G)

[[0.e+00 1.e-06 0.e+00]
 [0.e+00 0.e+00 0.e+00]
 [0.e+00 0.e+00 0.e+00]]


In [7]:
import pyvista as pv
pv.set_jupyter_backend("server")
# ============================================================
# 3D REFERENCE GEOMETRY

plotter = pv.Plotter(
    window_size=(1100, 800)
)

plotter.set_background("white")

# Unit sphere S^2

sphere = pv.Sphere(
    radius=1.0,
    theta_resolution=80,
    phi_resolution=80
)

plotter.add_mesh(
    sphere,
    color="lightgray",
    opacity=0.22,
    smooth_shading=True,
    specular=0.15,
    show_edges=False
)


# ------------------------------------------------------------
# Initial particle orientation p0
# ------------------------------------------------------------

p0_arrow = pv.Arrow(
    start=(0.0, 0.0, 0.0),
    direction=p0,
    tip_length=0.22,
    tip_radius=0.05,
    shaft_radius=0.018,
    scale=0.98
)

plotter.add_mesh(
    p0_arrow,
    color="green"
)

p0_point = pv.PolyData(
    p0.reshape(1, 3)
)

plotter.add_mesh(
    p0_point,
    color="green",
    point_size=9,
    render_points_as_spheres=True
)


# ------------------------------------------------------------
# Optical beam axis
# ------------------------------------------------------------

beam_arrow = pv.Arrow(
    start=(0.0, 0.0, 0.0),
    direction=beam_axis,
    tip_length=0.18,
    tip_radius=0.04,
    shaft_radius=0.012,
    scale=0.85
)

plotter.add_mesh(
    beam_arrow,
    color="orange"
)


# ------------------------------------------------------------
# Polarization direction
# ------------------------------------------------------------

polarization_arrow = pv.Arrow(
    start=(0.0, 0.0, 0.0),
    direction=e_pol,
    tip_length=0.18,
    tip_radius=0.04,
    shaft_radius=0.012,
    scale=0.85
)

plotter.add_mesh(
    polarization_arrow,
    color="purple"
)


# ------------------------------------------------------------
# Background velocity field
# ------------------------------------------------------------

grid = np.linspace(-1.0, 1.0, 11)

points = []
vectors = []

for a_grid in grid:
    for b_grid in grid:

        if plane == "xy":
            X = np.array([
                a_grid,
                b_grid,
                0.0
            ])

        elif plane == "xz":
            X = np.array([
                a_grid,
                0.0,
                b_grid
            ])

        elif plane == "yz":
            X = np.array([
                0.0,
                a_grid,
                b_grid
            ])

        else:
            raise ValueError(
                "plane must be 'xy', 'xz' or 'yz'"
            )

        u = G @ X

        points.append(X)
        vectors.append(u)


points = np.asarray(points)
vectors = np.asarray(vectors)

flow_mesh = pv.PolyData(points)

flow_mesh["velocity"] = vectors

flow_glyphs = flow_mesh.glyph(
    orient="velocity",
    scale="velocity",
    factor=0.30
)

plotter.add_mesh(
    flow_glyphs,
    color="black",
    opacity=0.80
)


# ------------------------------------------------------------
# Axes and bounds
# ------------------------------------------------------------

plotter.add_axes(
    line_width=3,
    xlabel="x",
    ylabel="y",
    zlabel="z"
)

plotter.show_bounds(
    grid="front",
    location="outer",
    all_edges=True,
    color="black",
    xtitle="px",
    ytitle="py",
    ztitle="pz",
    font_size=18
)


# ------------------------------------------------------------
# Text
# ------------------------------------------------------------

if plane == "xy":

    flow_text = (
        f"u = ({gamma:g} y, 0, 0)"
    )

elif plane == "xz":

    flow_text = (
        f"u = ({gamma:g} z, 0, 0)"
    )

elif plane == "yz":

    flow_text = "simple shear in yz"


plotter.add_text(
    "Reference geometry on S^2\n"
    f"{flow_text}\n"
    f"c = {aspect_ratio:.3f}, beta = {beta:.3f}",
    position="upper_left",
    font_size=16,
    color="black"
)


# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------

plotter.add_legend(
    labels=[
        ["initial orientation p0", "green"],
        ["beam axis", "orange"],
        ["polarization", "purple"],
        ["background flow", "black"]
    ],
    bcolor="white",
    border=False,
    face="circle",
    size=(0.22, 0.18)
)


# ------------------------------------------------------------
# Camera
# ------------------------------------------------------------

plotter.camera_position = "iso"
plotter.camera.zoom(1.15)

plotter.show()

Widget(value='<iframe src="http://localhost:52318/index.html?ui=P_0x1240ac3d0_0&reconnect=auto" class="pyvista…

# Step 1. Classical Jeffery orbit

The first case considers an elongated passive particle suspended in a simple shear flow. No swimming or optical effects are included. For a simple shear flow in the xy-plane 
$$ \mathbf{u} = (\dot{\gamma}y,0,0) $$

the velocity-gradient tensor ir 

$$
\mathbf{G} = \nabla\mathbf{u}=
\begin{pmatrix}
0 & \dot{\gamma} & 0\\
0 & 0 & 0\\
0 & 0 & 0
\end{pmatrix}.
$$
Then is decomosed into its symmetric and antisymmetric parts

$$
\mathbf{E} = \frac{1}{2}
\left(\mathbf{G}+\mathbf{G}^T\right)=
\begin{pmatrix}
0 & \dot{\gamma} & 0\\
\dot{\gamma} & 0 & 0\\
0 & 0 & 0
\end{pmatrix},
$$

$$
\mathbf{W} = \frac{1}{2}
\left(\mathbf{G}-\mathbf{G}^T\right)=
\begin{pmatrix}
0 & \dot{\gamma} & 0\\
-\dot{\gamma} & 0 & 0\\
0 & 0 & 0
\end{pmatrix}.
$$ 

The director follows the Jeffery´s equation:

$$ \dot{\mathbf{p}}= \mathbf{Wp}+\beta[\mathbf{Ep}-(\mathbf{p}^T\mathbf{Ep})\mathbf{p}] $$

## Step 2. Harmonic approximation near the trap center

In optical tweezers trapping potential, near the focus($x\approx0, y\approx0 \text{ and } z\approx0$), a gaussian beam can be well-approximated as harmonic:
$$ U(x,y,z)\approx-U_0+\frac{1}{2}k_xx^2+\frac{1}{2}k_yy^2+\frac{1}{2}k_zz^2$$

The trap stiffnesses $k_x, k_y \text{ and } k_z$ can be obtained from Taylor expanding the full Gaussian potential expression to the second order around the origin. Doing the taylor Taylor expansion we get:
$$ k_x =\frac{4U_0}{w_0^2},\qquad k_y=\frac{4U_0}{w_0^2},\qquad  k_z=\frac{2U_0}{z_R^2}$$

In [8]:
#Harmonic trap parameters
zR = jf.rayleigh_range(w0,wavelength) 

U0 = jf.trap_depth(alpha_scalar,laser_power,w0)

kx, ky, kz = jf.harmonic_stiffness_from_gaussian(U0,w0,zR)

print("Rayleigh range zR =", zR, "m")
print("Trap depth U0 =", U0, "J")

print()
print("Harmonic stiffnesses:")
print("kx =", kx, "N/m")
print("ky =", ky, "N/m")
print("kz =", kz, "N/m")

Rayleigh range zR = 0.008047112329891887 m
Trap depth U0 = 2.2789153901696853e-18 J

Harmonic stiffnesses:
kx = 3.6462646242714963e-09 N/m
ky = 3.6462646242714963e-09 N/m
kz = 7.03846685151165e-14 N/m


The optical force is obtained from the potential as 
$$\mathbf{F}=-\nabla U_{\mathrm{harm}}$$
Then for the harmonic potential
$$\mathbf{F}=(-kxx,-k_yy,-k_zz)$$
Meaning that the force always points toward the trap center. At the trap center $ \mathbf{R}=(0,0,0)$, the force will be 0; at small displacements it is generated a restoring force proportional to the displacement.

In [9]:
R_test = np.array([
    1.0e-6,
    -1.0e-6,
    1.0e-6
])

U_test = jf.harmonic_com_potential(R_test,kx,ky,kz)

F_test = jf.harmonic_com_force(R_test,kx,ky,kz)

print("R_test =", R_test, "m")
print("U_harm(R_test) =", U_test, "J")
print("F_opt(R_test) =", F_test, "N")

R_test = [ 1.e-06 -1.e-06  1.e-06] m
U_harm(R_test) = 3.6462998166057535e-21 J
F_opt(R_test) = [-3.64626462e-15  3.64626462e-15 -7.03846685e-20] N


## Intensity gradient torques 

Independent of any polarizability anisotropy, when L is comparable to $w_0$, tilting the rod pushes part of its length into lower-intensity regions. The potential is proportional to optical intensity of the beam, integrated over the particle volume:
$$U_{gradient}=-\frac{\alpha_V}{2}\int_V|E(r)|^2dV$$
Where $\alpha_V$ is the polarizability per unit volume. Using the harmonic approximation and knowing that for an induced dipole with scalar polarizability $\alpha$: 
$$U=-\frac{1}{2}\alpha|E|^2$$
we could equal:
$$-\frac{1}{2}\alpha|E|^2=-U_0+\frac{1}{2}k_xx^2+\frac{1}{2}k_yy^2+\frac{1}{2}k_zz^2$$
Clearing the field we get:
$$|E(x,y,z)|^2\approx\frac{2U_0}{\alpha}-\frac{1}{\alpha}(k_xx^2+\frac{1}{2}k_yy^2+\frac{1}{2}k_zz^2)$$

### Particle Geometry 

The particle is being modeled as a prolate spheroid with semi-major axis:
$$ a=\frac{L}{2}$$

and transverse semi-axes 
$$b=\frac{D}{2}$$

The long axis of the particle is represented by the unit director $\textbf{p}$. The orientation $\theta$ represents the angle from the z-axis, which corresponds the optical axis; $\theta=0$ means that the particle's long axis lies on the beam axis, whereas $\theta=\pi/2$ means that the particle's long axis lies entirely in the transverse xy-plane.

For a uniform ellipsoid, the normalized second spatial moment along a semi-axis of length is:
$$\frac{1}{V}\int_Vz^2dV=\frac{a^2}{5}$$

and along the transveres axis:
$$\frac{1}{V}\int_Vx^2dV=\frac{1}{V}\int_Vy^2dV=\frac{b^2}{5}$$

This quantities represent the mean squared distance of the particle volume from its center along each direction. Defining the normalized second-moment tensor as:
$$M_{ij}=\frac{1}{V}\int_Vx_ix_jdV$$
In 3-D the complete matrix is:
$$
M = \frac{1}{V}
\begin{pmatrix}
\int_Vx^2\text{ }dV & \int_Vxy\text{ }dV & \int_Vxz\text{ }dV\\
\int_Vyx\text{ }dV & \int_Vy^2\text{ }dV & \int_Vyz\text{ }dV\\
\int_Vzx\text{ }dV & \int_Vzy\text{ }dV & \int_Vz^2\text{ }dV
\end{pmatrix} =
\frac{1}{V}
\begin{pmatrix}
\int_Vx^2\text{ }dV & 0 & 0\\
0 & \int_Vy^2\text{ }dV & 0\\
0 & 0 & \int_Vz^2\text{ }dV
\end{pmatrix}
$$
Hence
$$
M_{body} =
\frac{1}{5}
\begin{pmatrix}
b^2 & 0 & 0\\
0 & b^2 & 0\\
0 & 0 & a^2
\end{pmatrix}
$$
It can be expressed relative to the z-axis as:
$$M=\frac{b^2}{5}\mathbf{I} + \frac{a^2-b^2}{5}\mathbf{pp}^T$$

Taking $M_{zz}$ and $p_z = \cos\theta$ we obtain:
$$\int_Vz^2\text{ }dV=\frac{1}{5}[b^2+(a^2-b^2)\cos^2\theta]$$
When  $\theta=0$ 
$$\int_Vz^2\text{ }dV=\frac{b^2}{5}\qquad \text{(lies in the beam axis)}$$
The total normalized second spatial moment is invariant under rotation, 
$$\frac{1}{V}\int_V(x^2+y^2+z^2)dV = \frac{a^2+2b^2}{5}\qquad\qquad\therefore\frac{1}{V}\int_V(x^2+y^2)dV = \frac{a^2+2b^2}{5}-\int_Vz^2\text{ }dV$$
Substituting we get:
$$\frac{1}{V}\int_V(x^2+y^2)dV =\frac{1}{5}[2b^2+(a^2-b^2)\sin^2\theta]$$

Since $k_x=k_y$, the we define $k_x=k_y=k_\perp$.
Substituing the fiel expesion:
$$U_{gradien}=-U_0+\frac{k_\perp}{2V}\int_V(x^2+y^2)dV+\frac{k_z}{2V}\int_V(z^2)dV$$
This gets:
$$ U_{\mathrm{gradient}}(\theta) = -U_0 + \frac{k_{\perp}}{10} \left[ 2b^2 + (a^2-b^2)\sin^2\theta \right] + \frac{k_z}{10} \left[ b^2 + (a^2-b^2)\cos^2\theta \right] $$

The terms that do not depend on $\theta$ can be considered as constants $C$, therefore
$$U_{gradient}=C+\frac{a^2-b^2}{10}[k_\perp\sin^2\theta+k_z\cos^2\theta]$$
Comparing with $\theta =0$ and $\theta = \pi/2$ we get:
$$\Delta U_{gradient}=\frac{a^2-b^2}{10}(k_z-k_\perp)$$
Since generally  $k_\perp>k_z$, the orientation parallel to the beam axis has lower energy ($U_{gradient}(\pi/2)> U_{gradient}(0) $, defining
$$k_{grad}=U_{gradient}(\pi/2)- U_{gradient}(0)=\frac{a^2-b^2}{10}(k_\perp-k_z) $$ 
or 
$$k_{grad}=\frac{L^2-D^2}{40}(k_\perp-k_z)$$

Assuiming the potential is sinusoidal, we can match to:
$$U(\theta)\approx -\frac{k_{grad}}{2}\cos\theta$$
and the gradient torque is 
$$\tau_{grad}(\theta)=-\frac{dU_{grad}}{d\theta}\approx- k_{grad}\sin$$

In [10]:
# ============================================================
# INTENSITY-GRADIENT TORQUE
# Harmonic approximation
# ============================================================



k_perp = 0.5 * (kx + ky)

kappa_grad_harmonic = (
    (a**2 - b**2)
    / 10.0
    * (k_perp - kz)
)

tau_grad_0 = (
    -kappa_grad_harmonic
    * np.sin(2.0 * theta0)
)

print(
    "kappa_grad =",
    kappa_grad_harmonic,
    "J"
)

print(
    "tau_grad(theta0) =",
    tau_grad_0,
    "N m"
)

kappa_grad = 4.648897655493803e-21 J
tau_grad(theta0) = -3.5612622156198725e-21 N m


### Orientation rate

The orientation rate ($\mathbf{p}_{grad}$) that will be added to the orientation rate obtained from the Jeffery's equation ($\mathbf{p}_{Jeff}$) will be obtained through the gradient torque ($\tau_{grad}$), which intrinsically wants to align the director with the z-axis. In a $\textbf{over-damped}$ system (low Reynolds number regime, rotational inertia is disregarded and al torques act at the same time) the gradient torque can be balance by viscous rotational drag,
$$\zeta_r\Omega_{grad}=\tau_{grad}$$
Where $\zeta_r$ is the rotational drag coefficient. Hence
$$\Omega_{grad}=\frac{\tau_{grad}}{\zeta_r}\qquad\therefore\qquad\Omega_{grad}=\frac{2k_{grad}}{\zeta_r}(\mathbf{p}\cdot\hat{\mathbf{z}})(\mathbf{p}\times\hat{\mathbf{z}})$$
Defining 
$$\xi=\frac{2k_{grad}}{\zeta_r}$$
The gradient angular velocity becomes:
$$\Omega_{grad}=\xi(\mathbf{p}\cdot\hat{\mathbf{z}})(\mathbf{p}\times\hat{\mathbf{z}})$$

The change in the orientation vector due to an angular velocity is:
$$\dot{\mathbf{p}}_{grad}=\Omega_{grad}\times\mathbf{p}$$
Hence
$$\dot{\mathbf{p}}_{grad}=\xi(\mathbf{p}\cdot\hat{\mathbf{z}})[(\mathbf{p}\times\hat{\mathbf{z}})\times \mathbf{p}]$$
Using the vector identity
$$(\mathbf{a}\times\hat{\mathbf{b}})\times \mathbf{c}=(\mathbf{a}\cdot\hat{\mathbf{a}})\mathbf{b}-(\mathbf{b}\cdot\hat{\mathbf{c}})\mathbf{a}$$
Then
$$(\mathbf{p}\times\hat{\mathbf{z}})\times \mathbf{p}=\hat{\mathbf{z}}-(\mathbf{p}\cdot\hat{\mathbf{z}})\mathbf{p}$$